## Install MLConfGen and other deps

In [ ]:
!pip install "mlconfgen[torch]==0.4.3"
!pip install py3Dmol

## Download the weights from HuggingFace
> https://huggingface.co/Membrizard/ml_conformer_generator

`edm_moi_chembl_15_39.pt`

`adj_mat_seer_chembl_15_39.pt`

## Lets create some variable linkers for a known PROTAC (Bavdegalutamide)
I've created this `.mol` files in Avogadro, but any 3D editor, or rdkit can be used instead.

In [ ]:
import py3Dmol

protac = "./protac_example.mol"
reference = "./protac_reference.mol"
fixed_fragment = "./protac_example_fragment.mol"

view = py3Dmol.view(width=900, height=300, viewergrid=(1,3))

# Initial Protac
view.addModel(open(protac).read(), 'mol', viewer=(0,0))

# A Reference we will use in Generation (it should be < 40 heavy atoms)
view.addModel(open(reference).read(), 'mol', viewer=(0,1))

# Fixed fragment, to set anchors
view.addModel(open(fixed_fragment).read(), 'mol', viewer=(0,2))


for i in range(3):
    view.zoomTo(viewer=(0,i))
    view.setStyle({'stick': {}}, viewer=(0,i))

view.show()

In [ ]:
from mlconfgen.inertial_fragment_matching import ff_inertial_fragment_matching

import time
import torch

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

from mlconfgen import evaluate_samples, MLConformerGenerator
from mlconfgen.utils.mol_split import extract_fragment
from mlconfgen.inertial_fragment_matching import ff_inertial_fragment_matching

RDLogger.DisableLog('rdApp.*')

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps:0")
else:
    device = torch.device("cpu")


# Initiate the generator
generator = MLConformerGenerator(
                              edm_weights="../../edm_moi_chembl_15_39.pt",
                              adj_mat_seer_weights="../../adj_mat_seer_chembl_15_39.pt",
                              device=device,
                              diffusion_steps=40,
                             )

# Prepare input molecules: Reference and fixed fragment
ref_mol = Chem.MolFromMolFile(reference)
ff_mol = Chem.MolFromMolFile(fixed_fragment)
print("Inertial Fragment Matching happening...")

N_SAMPLES = 20

# Run the inertial fragment matching generation pipeline
# You can opt for Geometry optimisation, but in that case the positions of the fixed parts will change
final_mols = ff_inertial_fragment_matching(
                                            fixed_fragment=ff_mol,
                                            n_samples=N_SAMPLES,
                                            generator=generator,
                                            reference_conformer=ref_mol,
                                            variance=1,
                                            predict_bonds=True,
                                            optimize_geometry=False,
                                            )

print("Inertial Fragment Matching happened!")


_, std_samples = evaluate_samples(ref_mol, final_mols)

# Display results
mols = []
legends = []
average_shape_similarity = 0
for sample in std_samples:
    mol = Chem.MolFromMolBlock(sample['mol_block'])
    mol = Chem.MolFromSmiles(Chem.MolToSmiles(mol))
    mol.SetProp("Shape_Tanimoto", str(sample['shape_tanimoto']))
    mols.append(mol)
    legends.append(f"Shape Similarity - {round(sample['shape_tanimoto'], 2)}")
    average_shape_similarity += round(sample['shape_tanimoto'], 2)

average_shape_similarity = average_shape_similarity / len(std_samples)
print(f"AVERAGE SHAPE SIMILARITY - {average_shape_similarity}")
print(f"VALID SAMPLES - {round(len(std_samples) / N_SAMPLES, 2) * 100} %")
    
Draw.MolsToGridImage(mols, legends=legends)

## Visualise the generated linkers in 3D alongside the PROTAC (highlighted with magenta)
> A small drift of the fixed fragments is expected and may be resolved by additional alignment

In [ ]:
def show_overlay_grid(
    reference_mol,
    candidate_mols,
    n_cols=3,
    width=300,
    height=300,
):

    def mol_to_block(mol):
        return Chem.MolToXYZBlock(mol)

    n_rows = (len(candidate_mols) + n_cols - 1) // n_cols

    view = py3Dmol.view(
        viewergrid=(n_rows, n_cols),
        width=width * n_cols,
        height=height * n_rows,
    )

    ref_block = mol_to_block(reference_mol)

    for i, cand in enumerate(candidate_mols):
        r = i // n_cols
        c = i % n_cols

        

        # Add reference (magenta)
        view.addModel(ref_block, "xyz", viewer=(r, c))
        view.setStyle(
            {"model": 0},
            {"stick": {"color": "magenta", "radius": 0.2}},
            viewer=(r, c),
        )

        #
        cand_block = mol_to_block(cand)
        view.addModel(cand_block, "xyz", viewer=(r, c))
        view.setStyle(
                {"model": 1},
                {"stick": {"radius": 0.2}},
                viewer=(r, c),
            )

        view.zoomTo(viewer=(r, c))

    return view

show_overlay_grid(Chem.MolFromMolFile(protac), final_mols)